In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

In [3]:
import os

print(os.path.exists(r"..\model\artifacts\feature_cols.pkl"))

True


In [6]:
df = pd.read_csv(r'..\Dataset\water_quality_cleandataversion.csv')
model = joblib.load(r"..\model\artifacts\best_model_tuned.pkl")
qt = joblib.load(r"..\model\artifacts\quantile_transformer.pkl")
FEATURE_COLS = joblib.load(r"..\model\artifacts\feature_cols.pkl")

print(f"Loaded cleaned data: {df.shape}")
print(f"Model expects {len(FEATURE_COLS)} features:", FEATURE_COLS)

Loaded cleaned data: (2617, 23)
Model expects 9 features: ['Temperature', 'DO_sqrt', 'pH', 'Conductivity_log', 'BOD_log', 'Nitrate_Nitrite_log', 'BOD_Temp_log', 'BOD_Conductivity_log', 'Nitrate_Temp']


d:\mini_water_predict\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator QuantileTransformer from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
# compare state-year averages for all features + target
BASE_COLS = ['Temperature', 'DO', 'pH', 'Conductivity', 'BOD', 'Nitrate_Nitrite']
TARGET_COL = 'Fecal_Coliform'

state_year = (
    df.groupby(['State_Name', 'Year'])[BASE_COLS + [TARGET_COL]]
    .mean()
    .reset_index()
)

print(f"State-year rows: {state_year.shape}")
state_year.head()

State-year rows: (137, 9)


,State_Name,Year,Temperature,DO,pH,Conductivity,BOD,Nitrate_Nitrite,Fecal_Coliform
0,ANDHRA PRADESH,2017,26.000000,6.350000,7.650000,815.500000,1.800000,2.260000,20.000000
1,ANDHRA PRADESH,2018,28.000000,6.250000,7.700000,962.500000,1.350000,2.000000,27.000000
2,ANDHRA PRADESH,2019,28.750000,5.275000,7.550000,997.500000,7.450000,2.555000,83.500000
3,ANDHRA PRADESH,2020,27.250000,5.600000,7.350000,586.500000,2.625000,2.240000,71.250000
4,ANDHRA PRADESH,2021,25.558824,5.644118,7.494118,838.147059,3.702941,1.675588,143.588235


In [8]:
# Helper extrapolate one feature for one state to a target year
def extrapolate_feature(state_df, feature, target_year):
    """Fit Year -> feature on available years, predict at target_year."""
    sub = state_df.dropna(subset=[feature])
    if len(sub) < 2:
        # Not enough history to fit a trend fall back to last known value
        return sub[feature].iloc[-1] if len(sub) == 1 else np.nan
    X = sub[['Year']].values
    y = sub[feature].values
    reg = LinearRegression().fit(X, y)
    pred = reg.predict([[target_year]])[0]
    return max(pred, 0)  # water quality features can't be negative

In [9]:
# Build synthetic state-year row for a target year
def build_synthetic_year(state_year_df, target_year, state_freq_mapping, water_type_mode):
    """
    state_year_df : state_year table filtered to years < target_year
    Returns a dataframe with one row per state, containing extrapolated
    BASE_COLS, ready for feature engineering.
    """
    rows = []
    for state in state_year_df['State_Name'].unique():
        state_hist = state_year_df[state_year_df['State_Name'] == state].sort_values('Year')
        row = {'State_Name': state, 'Year': target_year}
        for feat in BASE_COLS:
            row[feat] = extrapolate_feature(state_hist, feat, target_year)
        rows.append(row)

    synth_df = pd.DataFrame(rows)
    # Water_Body_Type: use the most common type historically per state (mode)
    synth_df['Water_Body_Type'] = synth_df['State_Name'].map(water_type_mode)
    return synth_df

In [10]:
# Feature engineering MUST mirror the ML notebook exactly ===
def apply_feature_engineering(synth_df, state_freq_mapping):
    d = synth_df.copy()

    d['DO_sqrt'] = np.sqrt(d['DO'].clip(lower=0))
    d['Conductivity_log'] = np.log1p(d['Conductivity'])
    d['BOD_log'] = np.log1p(d['BOD'])
    d['Nitrate_Nitrite_log'] = np.log1p(d['Nitrate_Nitrite'])

    d['BOD_Temp'] = d['BOD'] * d['Temperature']
    d['BOD_Conductivity'] = d['BOD'] * d['Conductivity']
    d['Nitrate_Temp'] = d['Nitrate_Nitrite'] * d['Temperature']

    d['BOD_Temp_log'] = np.log1p(d['BOD_Temp'].clip(lower=0))
    d['BOD_Conductivity_log'] = np.log1p(d['BOD_Conductivity'].clip(lower=0))

    d['Is_Monsoon'] = d['Year'].astype(int).apply(
        lambda x: 1 if x in [2017, 2018, 2019, 2020, 2021, 2022, 2023] else 0
    )

    d['State_freq'] = d['State_Name'].map(state_freq_mapping).fillna(0)

    d = pd.get_dummies(d, columns=['Water_Body_Type'])
    # Ensure every dummy column the model expects exists, even if absent this round
    for col in FEATURE_COLS:
        if col.startswith('Water_Body_Type_') and col not in d.columns:
            d[col] = 0

    return d

In [12]:
# 1. Build mappings
state_freq_mapping = df['State_Name'].value_counts(normalize=True).to_dict()

water_type_mode = (
    df.groupby('State_Name')['Water_Body_Type']
      .agg(lambda x: x.mode().iloc[0])
      .to_dict()
)

# 2. Then one-hot encode
df = pd.get_dummies(
    df,
    columns=['Water_Body_Type'],
    drop_first=True
)

In [13]:
# Walk-forward loop
results = []

all_years = sorted(state_year['Year'].unique())
forecast_years = list(range(min(all_years) + 1, 2024))  # e.g. 2018..2023

for target_year in forecast_years:
    train_hist = state_year[state_year['Year'] < target_year]

    if train_hist.empty:
        continue

    synth = build_synthetic_year(train_hist, target_year, state_freq_mapping, water_type_mode)
    synth_fe = apply_feature_engineering(synth, state_freq_mapping)

    X_pred = synth_fe.reindex(columns=FEATURE_COLS, fill_value=0)

    pred_qt = model.predict(X_pred)
    pred_raw = qt.inverse_transform(pred_qt.reshape(-1, 1)).flatten().clip(0)

    synth['Predicted_FC'] = pred_raw
    synth['Year'] = target_year

    # Attach real value if it exists (years <= 2022)
    real_lookup = (
        state_year[state_year['Year'] == target_year]
        .set_index('State_Name')
        .get('Fecal_Coliform') if 'Fecal_Coliform' in state_year.columns else None
    )
    # NOTE: Fecal_Coliform wasn't in BASE_COLS above — see Cell 3 fix below

    synth['Safety'] = synth['Predicted_FC'].apply(lambda x: 'Safe' if x <= 50 else 'Not Safe')
    results.append(synth[['State_Name', 'Year', 'Predicted_FC', 'Safety']])

forecast_all = pd.concat(results, ignore_index=True)
print(f"Forecast rows: {forecast_all.shape}")
forecast_all.head(10)

Forecast rows: (147, 4)


,State_Name,Year,Predicted_FC,Safety
0,ANDHRA PRADESH,2018,28.578516,Safe
1,ASSAM,2018,4166.310059,Not Safe
2,BIHAR,2018,89.761604,Not Safe
3,CHANDIGARH,2018,22975.289062,Not Safe
4,CHHATTISGARH,2018,515.000000,Not Safe
5,GOA,2018,2435.370850,Not Safe
6,GUJARAT,2018,2918.553467,Not Safe
7,HARYANA,2018,181.000000,Not Safe
8,HIMACHAL PRADESH,2018,99.651169,Not Safe
9,KARNATAKA,2018,9657.166992,Not Safe


In [17]:
print(forecast_all.columns.tolist())

['State_Name', 'Year', 'Predicted_FC', 'Safety']


In [18]:
forecast_all = forecast_all.merge(
    df[['State_Name', 'Year', 'Fecal_Coliform']],
    on=['State_Name', 'Year'],
    how='left'
)

forecast_all.rename(
    columns={'Fecal_Coliform': 'Real_FC'},
    inplace=True
)

print(forecast_all.head())

       State_Name  Year  Predicted_FC    Safety  Real_FC
0  ANDHRA PRADESH  2018     28.578516      Safe     27.0
1           ASSAM  2018   4166.310059  Not Safe   1351.0
2           ASSAM  2018   4166.310059  Not Safe    181.0
3           ASSAM  2018   4166.310059  Not Safe   1550.0
4           ASSAM  2018   4166.310059  Not Safe   1751.0


In [20]:
forecast_all.columns = forecast_all.columns.str.strip()

print(forecast_all.columns.tolist())

['State_Name', 'Year', 'Predicted_FC', 'Safety', 'Real_FC']


In [22]:
# Keep rows that have real observations
eval_df = forecast_all.dropna(subset=['Real_FC']).copy()

# Compute absolute error
eval_df['Abs_Error'] = (
    eval_df['Predicted_FC'] - eval_df['Real_FC']
).abs()

# Yearly summary
accuracy_summary = (
    eval_df.groupby('Year')
    .agg(
        MAE=('Abs_Error', 'mean'),
        States_Evaluated=('Real_FC', 'size')
    )
    .reset_index()
)

print(accuracy_summary)

   Year           MAE  States_Evaluated
0  2018  20924.282433               322
1  2019  24925.190742               433
2  2020  34813.969498               461
3  2021  34238.778637               515
4  2022  16169.305884               546


In [26]:
from pathlib import Path

out_path = Path("mini_water_predict/Dataset")
out_path.mkdir(parents=True, exist_ok=True)

forecast_all.to_csv(out_path / "forecast_all_years.csv", index=False)